# Customer Table Structure


In [0]:
%sql
Create table Customer (
    Customer_id string not null,
    Customer_name string,
    email string,
    phone string,
    billing_address1 string,
    billing_address2 string,
    city string,
    postalcode string,
    country string,
    created_at timestamp not null ,
    updated_at timestamp );
    

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5208616380218452>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'Create table Customer (\n    Customer_id string not null,\n    Customer_name string,\n    email string,\n    phone string,\n    billing_address1 string,\n    billing_address2 string,\n    city string,\n    postalcode string,\n    country string,\n    created_at timestamp not null ,\n    updated_at timestamp );\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in 

## Importing Libraries

In [0]:
# Requirements

%pip install fastavro faker

# Step 1 — Import Libraries

from faker import Faker
import random
from datetime import datetime
import json
from fastavro import writer, parse_schema
import pandas as pd
from pyspark.sql import SparkSession
import pyarrow as pa
import pyarrow.parquet as pq
import os

# Step 2 — Create Faker and Spark

fake = Faker()
spark = SparkSession.builder.getOrCreate()

# Step 3 — Helper: Random Rows

def rnd_int(n=999999):
    return random.randint(1, n)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 107.8 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## Genrating data to send to the source

In [0]:
def generate_customer_csv(n):
    data = []
    for _ in range(n):
        data.append({
            "customer_id": rnd_int(),
            "first_name": fake.first_name(),
            "last_name": fake.last_name(),
            "email": fake.email(),
            "phone": fake.phone_number(),
            "birth_date": fake.date_of_birth().isoformat(),
            "created_at": datetime.now().isoformat()
        })

    df = pd.DataFrame(data)
    df_s = spark.createDataFrame(df)
    df_s.write\
         .mode("append") \
         .option("header", "true") \
         .csv("abfss://source@project1ps.dfs.core.windows.net/customer/")

## genrating data

In [0]:

generate_customear_csv(10)

---------------------------------------------------------------------------
UnknownException                          Traceback (most recent call last)
File <command-6939383657975511>, line 2
----> 2 generate_customer_csv(10)

File <command-6939383657975509>, line 19, in generate_customer_csv(n)
     14 df = pd.DataFrame(data)
     15 df_s = spark.createDataFrame(df)
     16 df_s.write\
     17      .mode("append") \
     18      .option("header", "true") \
---> 19      .csv("abfss://source@project1ps.dfs.core.windows.net/customer/")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:831, in DataFrameWriter.csv(self, path, mode, compression, sep, quote, escape, header, nullValue, escapeQuotes, quoteAll, dateFormat, timestampFormat, ignoreLeadingWhiteSpace, ignoreTrailingWhiteSpace, charToEscapeQuoteEscaping, encoding, emptyValue, lineSep)
    812 self.mode(mode)
    813 self._set_opts(
    814     compression=compression,
    815     sep=sep,
   (..